# 🔍 Validação da View de Status dos Pedidos

## 📦 Análise de Pedidos — Olist E-commerce

Análise dos pedidos por status, volume e evolução temporal,
utilizando a data de compra como referência.

### Célula 2 — Imports e carregamento:

In [8]:
import pandas as pd

pd.set_option('display.float_format', '{:.2f}'.format)

pedidos = pd.read_csv("../dados/pedidos_limpo.csv", parse_dates=[
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
])

clientes = pd.read_csv("../dados/clientes_limpo.csv")

pedidos.shape

(99441, 8)

## 📊 Análise de Status dos Pedidos

In [9]:
# value_counts() — conta a frequência de cada valor único em uma coluna

pedidos['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

## 📊 Validando vw_status_pedidos

In [10]:
# Extraindo ano e mês da data de compra
# dt.to_period('M') — converte para período mensal (2017-01, 2017-02...)

def get_status_pedidos(pedidos, clientes):

    status_traducao = {
        'delivered':   'Entregue',
        'shipped':     'Em Transporte',
        'canceled':    'Cancelado',
        'unavailable': 'Indisponível',
        'invoiced':    'Faturado',
        'processing':  'Em Processamento',
        'created':     'Criado',
        'approved':    'Aprovado'
    }

    df = pedidos.merge(
        clientes[['customer_id', 'customer_state']],
        on='customer_id',
        how='left'
    )

    df['status_pt'] = df['order_status'].map(status_traducao)
    df['ano'] = df['order_purchase_timestamp'].dt.year
    df['mes'] = df['order_purchase_timestamp'].dt.month
    df['ano_mes'] = df['order_purchase_timestamp'].dt.to_period('M').astype(str)

    resultado = (
        df.groupby([
            'customer_state',
            'ano',
            'mes',
            'ano_mes',
            'status_pt'
        ])
        .size()
        .reset_index(name='quantidade')
    )

    total_mes = (
        resultado
        .groupby(['customer_state', 'ano_mes'])['quantidade']
        .transform('sum')
    )

    resultado['percentual'] = (
        resultado['quantidade'] / total_mes * 100
    ).round(2)

    return resultado

In [11]:
df_status = get_status_pedidos(pedidos, clientes)
df_status.head()

,customer_state,ano,mes,ano_mes,status_pt,quantidade,percentual
0,AC,2017,1,2017-01,Entregue,2,100.00
1,AC,2017,2,2017-02,Entregue,3,100.00
2,AC,2017,3,2017-03,Entregue,2,100.00
3,AC,2017,4,2017-04,Entregue,5,100.00
4,AC,2017,5,2017-05,Entregue,8,100.00


### 🧪 Testando a View

A função `vw_status_pedidos` é importada diretamente da pasta `views`
para validar a transformação utilizada pelo dashboard.

A análise considera o estado do cliente para permitir a segmentação
dos pedidos por localização geográfica.

In [12]:
import sys
sys.path.append('..')

from views.vw_status_pedidos import get_status_pedidos

df_status = get_status_pedidos(pedidos, clientes)
df_status.head()

,customer_state,ano,mes,ano_mes,status_pt,quantidade,percentual
0,AC,2017,1,2017-01,Entregue,2,100.00
1,AC,2017,2,2017-02,Entregue,3,100.00
2,AC,2017,3,2017-03,Entregue,2,100.00
3,AC,2017,4,2017-04,Entregue,5,100.00
4,AC,2017,5,2017-05,Entregue,8,100.00
